In [1]:
import kagglehub, duckdb, os, shutil

src = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
os.makedirs("data", exist_ok=True)
for f in os.listdir(src):
    shutil.copy(os.path.join(src, f), "data")
print(os.listdir("data"))

c:\Users\박현아\Desktop\조폐공사\olist-marketplace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 42.6M/42.6M [00:04<00:00, 9.79MB/s]

Extracting files...


['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [2]:
files = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
}
con = duckdb.connect()
for name, f in files.items():
    con.sql(f"CREATE VIEW {name} AS SELECT * FROM read_csv_auto('data/{f}')")

def q(sql):
    return con.sql(sql).df()

q("SELECT * FROM orders LIMIT 5")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [5]:
q("SELECT COUNT(*) FROM ORDERS")

,count_star()
0,99441


In [6]:
q("select count(*) AS a, count(distinct order_id) as b from orders")

,a,b
0,99441,99441


In [7]:
q("select count(*) as a, count(distinct customer_id) as b from customers")


,a,b
0,99441,99441


In [8]:
q("select count(*) as a, count(distinct customer_unique_id) as b from customers")


,a,b
0,99441,96096


In [9]:
q("select count(*) from items")

,count_star()
0,112650


In [10]:
q("select count(*) as a, count(distinct order_id) as b from items")

,a,b
0,112650,98666


In [13]:
q("select order_status, count(*) as 건수 from orders group by order_status order by 건수 desc")

,order_status,건수
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [14]:
q("select order_status, count(*) as 건수 from orders where order_id not in (select order_id from items) group by order_status order by 건수 desc")

,order_status,건수
0,unavailable,603
1,canceled,164
2,created,5
3,invoiced,2
4,shipped,1


In [16]:
q("select count(*) as a, count(distinct order_id) as b from payments")

,a,b
0,103886,99440


In [15]:
q("""
select payment_type, count(*) as 건수
from payments
group by payment_type
order by 건수 desc
""")

,payment_type,건수
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3


In [17]:
q("""
select *
from orders
where order_id not in (select order_id from payments)
""")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [18]:
q("""
select 결제횟수, count(*) as 주문수
from(
    select order_id, count(*) as 결제횟수
    from payments
    group by order_id
)
group by 결제횟수
order by 결제횟수
""")

,결제횟수,주문수
0,1,96479
1,2,2382
2,3,301
3,4,108
4,5,52
5,6,36
6,7,28
7,8,11
8,9,9
9,10,5


In [22]:
q("""
select order_id,
    sum(price) as 상품가,
    sum(freight_value) as 배송비
from items
group by order_id
limit 5
""")

,order_id,상품가,배송비
0,84499df9226b91d3b5817d5e6651a143,139.80,35.48
1,844bfdffbe80e8ac61135bc633f2dce1,60.89,17.53
2,845b62fbbe4d5781455b7da5d4989afa,26.89,18.29
3,84630355d3f0a8006b528d597b29e854,289.00,21.21
4,846de1a3e7ca6bf29333e1db02040ab6,258.00,32.68


In [24]:
q("""
select order_id,
    sum(payment_value) as 결제금액
from payments
group by order_id
limit 5
""")

,order_id,결제금액
0,1ffb3c1929b16d9c1aec1958e11b3e9b,166.04
1,3689194c14ad4e2e7361ebd1df0e77b0,153.27
2,a5f860aaf6a3c35399495aefe92d363f,75.75
3,31f8013402a0749b3ab433b1aee11754,42.19
4,886b114d034f4ac1d39d964c1b2a8182,2027.16


In [25]:
q("""
with 주문금액 as (
    select order_id,
           sum(price + freight_value) as 주문합계
    from items
    group by order_id
),
결제금액 as (
    select order_id,
           sum(payment_value) as 결제합계
    from payments
    group by order_id
)
select a.order_id,
       a.주문합계,
       b.결제합계,
       round(b.결제합계 - a.주문합계, 2) as 차이
from 주문금액 a
join 결제금액 b on a.order_id = b.order_id
order by abs(b.결제합계 - a.주문합계) desc
limit 10
""")

,order_id,주문합계,결제합계,차이
0,ce6d150fb29ada17d2082f4847107665,1403.66,1586.47,182.81
1,6e5fe7366a2e1bfbf3257dba0af1267f,287.91,406.92,119.01
2,70b742795bc441e94a44a084b6d9ce7a,466.93,578.82,111.89
3,996c7e73600ad3723e8627ab7bef81e4,587.90,664.43,76.53
4,70b7e94ea46d3e8b5bc12a50186edaf0,213.15,274.84,61.69
5,bc2c82b0ef78d2252b6176d1972db7c9,242.01,303.02,61.01
6,af9ffff2ce6b3defd34fd4c78857a379,413.17,466.97,53.80
7,262118ce178bb3e4590a3adcf6d62e6b,177.74,126.12,-51.62
8,bfdb5bbb06458d600a33d61f5f287472,348.93,394.36,45.43
9,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,254.45,293.89,39.44


In [29]:
q("""
with 주문금액 as (
    select order_id,
           sum(price + freight_value) as 주문합계
    from items
    group by order_id
),
결제금액 as (
    select order_id,
           sum(payment_value) as 결제합계
    from payments
    group by order_id
)
select count(*) as 불일치건수
from 주문금액 a
join 결제금액 b on a.order_id = b.order_id
where abs(b.결제합계 - a.주문합계) > 0.01
""")

,불일치건수
0,380


In [30]:
q("""
with 주문금액 as (
    select order_id, sum(price + freight_value) as 주문합계
    from items group by order_id
),
결제금액 as (
    select order_id, sum(payment_value) as 결제합계
    from payments group by order_id
)
select case when b.결제합계 > a.주문합계 then '결제가 더 많음'
            else '결제가 더 적음' end as 구분,
       count(*) as 건수,
       round(min(abs(b.결제합계 - a.주문합계)), 2) as 최소차이,
       round(max(abs(b.결제합계 - a.주문합계)), 2) as 최대차이,
       round(sum(abs(b.결제합계 - a.주문합계)), 2) as 차이합계
from 주문금액 a
join 결제금액 b on a.order_id = b.order_id
where abs(b.결제합계 - a.주문합계) > 0.01
group by 구분
""")

,구분,건수,최소차이,최대차이,차이합계
0,결제가 더 적음,89,0.01,51.62,199.58
1,결제가 더 많음,291,0.01,182.81,3070.41


In [34]:
q("""
with 주문금액 as (
    select order_id, sum(price + freight_value) as 주문합계
    from items group by order_id
),
결제금액 as (
    select order_id, sum(payment_value) as 결제합계
    from payments group by order_id
),
불일치 as (
    select a.order_id
    from 주문금액 a
    join 결제금액 b on a.order_id = b.order_id
    where abs(b.결제합계 - a.주문합계) > 0.01
)
select p.payment_type, count(*) as 건수,
round(avg(p.payment_installments), 1) as 평균할부
from payments p join 불일치 u on p.order_id = u.order_id
group by p.payment_type
""")

,payment_type,건수,평균할부
0,voucher,11,1.0
1,credit_card,351,5.8
2,boleto,29,1.0
3,debit_card,10,1.0


In [35]:
q("""
select payment_type,
       count(*) as 건수,
       round(avg(payment_installments), 1) as 평균할부
from payments
group by payment_type
order by 건수 desc
""")

,payment_type,건수,평균할부
0,credit_card,76795,3.5
1,boleto,19784,1.0
2,voucher,5775,1.0
3,debit_card,1529,1.0
4,not_defined,3,1.0


In [37]:
q("""
select i.seller_id,
    sum(i.price) as 매출
from items i
join orders o on i.order_id = o.order_id
where o.order_status = 'delivered'
group by i.seller_id
order by 매출 desc
limit 10
""")

,seller_id,매출
0,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93
1,53243585a1d6dc2643021fd1853d8905,217940.44
2,4a3ca9315b744ce9f8e9374361493884,196882.12
3,fa1c13f2614d7b5c4749cbc52fecda94,190917.14
4,7c67e1448b00f6e969d365cea6b010ab,186570.05
5,7e93a43ef30c4f03f38b393420bc753a,165981.49
6,da8622b14eb17ae2831f4ac5b9dab84a,159816.87
7,7a67c85e85bb2ce8582c35f2203ad736,139658.69
8,1025f0e2d44d7041d6cf58b6550e0bfa,138208.56
9,955fee9216a65b617aa5c0531780ce60,131836.71


In [39]:
q("""
select count(distinct i.seller_id) as 셀러수,
       sum(i.price) as 총매출
from items i
join orders o on i.order_id = o.order_id
where o.order_status = 'delivered'
""")

,셀러수,총매출
0,2970,1.322150e+07


In [40]:
q("""
with 셀러매출 as (
    select i.seller_id, sum(i.price) as 매출
    from items i
    join orders o on i.order_id = o.order_id
    where o.order_status = 'delivered'
    group by i.seller_id
),
순위 as (
    select seller_id, 매출,
           row_number() over (order by 매출 desc) as 순위,
           count(*) over () as 전체셀러수,
           sum(매출) over () as 총매출
    from 셀러매출
)
select case when 순위 <= 전체셀러수 * 0.01 then '상위 1%'
            when 순위 <= 전체셀러수 * 0.05 then '상위 1~5%'
            when 순위 <= 전체셀러수 * 0.10 then '상위 5~10%'
            when 순위 <= 전체셀러수 * 0.20 then '상위 10~20%'
            else '하위 80%' end as 구간,
       count(*) as 셀러수,
       round(sum(매출), 0) as 구간매출,
       round(sum(매출) / max(총매출) * 100, 1) as 매출비중
from 순위
group by 구간
order by 매출비중 desc
""")

,구간,셀러수,구간매출,매출비중
0,상위 1~5%,119,3625749.0,27.4
1,상위 1%,29,3373224.0,25.5
2,하위 80%,2376,2341838.0,17.7
3,상위 10~20%,297,2006583.0,15.2
4,상위 5~10%,149,1874103.0,14.2


In [41]:
q("""
select count(*) as 전체,
       sum(case when order_delivered_customer_date > order_estimated_delivery_date
                then 1 else 0 end) as 지연건수,
       round(avg(case when order_delivered_customer_date > order_estimated_delivery_date
                      then 1.0 else 0 end) * 100, 1) as 지연율
from orders
where order_status = 'delivered'
""")


,전체,지연건수,지연율
0,96478,7826.0,8.1


In [42]:
q("select count(*) from reviews")

,count_star()
0,99224


In [43]:
q("select count(*) as a, count(distinct order_id) as b from reviews")

,a,b
0,99224,98673


In [44]:
q("select count(*) as a, count(distinct review_id) as b from reviews")

,a,b
0,99224,98410


In [45]:
q("""
with 주문리뷰 as (
    select order_id, avg(review_score) as 리뷰점수
    from reviews
    group by order_id
)
select case when o.order_delivered_customer_date > o.order_estimated_delivery_date
            then '지연' else '정시' end as 구분,
       count(*) as 건수,
       round(avg(r.리뷰점수), 2) as 평균점수,
       round(avg(case when r.리뷰점수 <= 2 then 1.0 else 0 end) * 100, 1) as 불만비율
from orders o
join 주문리뷰 r on o.order_id = r.order_id
where o.order_status = 'delivered'
group by 구분
""")

,구분,건수,평균점수,불만비율
0,정시,88171,4.29,9.2
1,지연,7661,2.57,54.0


In [46]:
q("""
with 셀러주문 as (
    select distinct i.seller_id, i.order_id,
           case when o.order_delivered_customer_date > o.order_estimated_delivery_date
                then 1 else 0 end as 지연
    from items i
    join orders o on i.order_id = o.order_id
    where o.order_status = 'delivered'
),
셀러매출 as (
    select i.seller_id, sum(i.price) as 매출
    from items i
    join orders o on i.order_id = o.order_id
    where o.order_status = 'delivered'
    group by i.seller_id
)
select s.seller_id,
       round(m.매출, 0) as 매출,
       count(*) as 주문수,
       round(avg(s.지연) * 100, 1) as 지연율
from 셀러주문 s
join 셀러매출 m on s.seller_id = m.seller_id
group by s.seller_id, m.매출
order by 매출 desc
limit 20
""")

,seller_id,매출,주문수,지연율
0,4869f7a5dfa277a7dca6462dcf3b52b2,226988.0,1124,11.6
1,53243585a1d6dc2643021fd1853d8905,217940.0,348,4.3
2,4a3ca9315b744ce9f8e9374361493884,196882.0,1772,11.0
3,fa1c13f2614d7b5c4749cbc52fecda94,190917.0,578,10.2
4,7c67e1448b00f6e969d365cea6b010ab,186570.0,973,10.1
5,7e93a43ef30c4f03f38b393420bc753a,165981.0,319,5.6
6,da8622b14eb17ae2831f4ac5b9dab84a,159817.0,1311,7.6
7,7a67c85e85bb2ce8582c35f2203ad736,139659.0,1145,5.9
8,1025f0e2d44d7041d6cf58b6550e0bfa,138209.0,910,10.1
9,955fee9216a65b617aa5c0531780ce60,131837.0,1261,7.8
